# Deep Q-Learning on Atari Pong: training on Colab

Runs a member's 10 hyperparameter experiments (Task 3) and the final long training run on a Colab GPU.

1. **Runtime > Change runtime type > T4 GPU**, then save.
2. Set `MEMBER` below and run the cells from top to bottom.
3. The project lives in your Google Drive (`MyDrive/deep-q-learning`), so trained models survive a disconnect.
   If Colab disconnects, run all cells again: finished experiments are skipped.

In [ ]:
MEMBER = "shima"  # "shima" or "gaddiel" (see tools/experiments.py)
REPO_URL = "https://github.com/supserrr/deep-q-learning.git"

## 1. Get the project onto Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/deep-q-learning"
if os.path.exists(PROJECT):
    !git -C {PROJECT} pull
else:
    !git clone {REPO_URL} {PROJECT}
%cd {PROJECT}

## 2. Install the packages with uv

The environment goes on Colab's local disk (`/content/.venv`), which is much faster than Drive.

In [ ]:
!command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
os.environ["UV_PROJECT_ENVIRONMENT"] = "/content/.venv"
os.environ["MPLBACKEND"] = "Agg"  # Colab's default plot backend is not installed in our environment
!uv sync
!uv run python -c "import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, switch the runtime to T4 GPU')"

## 3. Run the 10 experiments

Each one trains for 200,000 steps, two at a time, which takes about 33 minutes per pair on a T4. Every finished game prints one line.

In [ ]:
!uv run tools/experiments.py --member {MEMBER} --parallel 2

## 4. Results table and training plots

Copy this table into your `results/<Name>_Hyperparameter_experiment_results.md` and write what you observed for each experiment.

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(open(f"models/{MEMBER}/results_table.md").read()))
for number in range(1, 11):
    plot = f"models/{MEMBER}/exp{number:02d}/training_plot.png"
    if os.path.exists(plot):
        print(f"Experiment {number}")
        display(Image(plot, width=700))

## 5. Train the best configuration for longer

Picks the experiment with the highest average reward over its last 10 games and trains it for 800,000 steps. This is the final agent.

In [ ]:
!uv run tools/experiments.py --member {MEMBER} --table --best-steps 800000
display(Image(f"models/{MEMBER}/best/training_plot.png", width=700))

## 6. Save the final model as `dqn_model.pth` (used by `play.py`)

In [ ]:
!cp models/{MEMBER}/best/dqn_model.pth dqn_model.pth
print("Saved. Everything is in Google Drive: MyDrive/deep-q-learning")